# 8. Reflexion Agent (Self-Critique Loop)
**Industry:** Consulting Services

Build an agent that answers a client question/brief, critiques its own answer, and revises it — repeating until a quality threshold or max iterations is reached.

In [ ]:
!pip install langgraph langchain langchain-google-genai pydantic

In [ ]:
from typing import TypedDict
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

class Critique(BaseModel):
    feedback: str = Field(description="Constructive criticism of the answer")
    is_good_enough: bool = Field(description="True if the answer meets professional consulting standards, False otherwise")

critique_llm = llm.with_structured_output(Critique)

class ReflexionState(TypedDict):
    question: str
    draft: str
    feedback: str
    iterations: int

def draft_node(state: ReflexionState):
    if state.get("feedback"):
        prompt = f"Revise this draft based on the feedback.\n\nDraft: {state['draft']}\nFeedback: {state['feedback']}"
    else:
        prompt = f"Draft an initial consulting answer for: {state['question']}"
    response = llm.invoke(prompt)
    return {"draft": response.content, "iterations": state.get("iterations", 0) + 1}

def critique_node(state: ReflexionState):
    prompt = f"Critique this consulting strategy.\nQuestion: {state['question']}\nDraft: {state['draft']}"
    eval = critique_llm.invoke(prompt)
    return {"feedback": eval.feedback, "is_good_enough": eval.is_good_enough}

def route(state: ReflexionState):
    if state.get("is_good_enough") or state["iterations"] >= 3:
        return END
    return "draft_node"

workflow = StateGraph(ReflexionState)
workflow.add_node("draft_node", draft_node)
workflow.add_node("critique_node", critique_node)

workflow.add_edge(START, "draft_node")
workflow.add_edge("draft_node", "critique_node")
workflow.add_conditional_edges("critique_node", route, ["draft_node", END])

app = workflow.compile()

inputs = {"question": "Outline a market-entry strategy for a new EV brand in India.", "iterations": 0}
for event in app.stream(inputs):
    for k, v in event.items():
        if k == "draft_node":
            print(f"\n[Draft Iteration {v['iterations']}]:\n", v['draft'][:200], "...")
        elif k == "critique_node":
            print(f"\n[Critique]:\n", v['feedback'])
            print("Pass?", v['is_good_enough'])